# 🎯 تحدي كشف الشماغ - Dal Shemagh Detection Challenge

## 📋 نظرة عامة:
هذا Notebook لتدريب نموذج YOLOv8 للمسابقة

### المهام:
1. ✅ كشف الرأس (Head Detection)
2. ✅ كشف الشماغ (Shemagh Detection)
3. ✅ تحديد هل الشماغ في المكان الصحيح

### Classes:
- **Class 0**: Head (رأس)
- **Class 1**: Shemagh (شماغ)

In [1]:
# تثبيت ultralytics (YOLOv8)
!pip install ultralytics -q

print("✅ تم تثبيت المكتبات بنجاح!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00a 0:00:01
✅ تم تثبيت المكتبات بنجاح!


In [2]:
import os
import shutil
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
import glob
from tqdm import tqdm

print("✅ تم استيراد المكتبات بنجاح!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ تم استيراد المكتبات بنجاح!


In [3]:
# مسارات البيانات في Kaggle - المسار الصحيح المُعدّل!
BASE_INPUT = "/kaggle/input/competitions/dal-shemagh-detection-challenge"

# مسارات البيانات الأصلية - البنية الحقيقية
TRAIN_IMAGES_SOURCE = f"{BASE_INPUT}/images/train"
TRAIN_LABELS_SOURCE = f"{BASE_INPUT}/labels/train"
TEST_IMAGES_SOURCE = f"{BASE_INPUT}/images/test"

# التحقق من وجود البيانات
print("🔍 التحقق من المسارات...")
print(f"   صور التدريب: {'✅' if os.path.exists(TRAIN_IMAGES_SOURCE) else '❌'} {TRAIN_IMAGES_SOURCE}")
print(f"   Labels التدريب: {'✅' if os.path.exists(TRAIN_LABELS_SOURCE) else '❌'} {TRAIN_LABELS_SOURCE}")
print(f"   صور الاختبار: {'✅' if os.path.exists(TEST_IMAGES_SOURCE) else '❌'} {TEST_IMAGES_SOURCE}")

# مسارات YOLO Dataset
WORKING_DIR = "/kaggle/working"
DATASET_DIR = f"{WORKING_DIR}/yolo_dataset"
TRAIN_IMAGES_DIR = f"{DATASET_DIR}/images/train"
VAL_IMAGES_DIR = f"{DATASET_DIR}/images/val"
TRAIN_LABELS_DIR = f"{DATASET_DIR}/labels/train"
VAL_LABELS_DIR = f"{DATASET_DIR}/labels/val"

# إنشاء المجلدات
for directory in [TRAIN_IMAGES_DIR, VAL_IMAGES_DIR, TRAIN_LABELS_DIR, VAL_LABELS_DIR]:
    os.makedirs(directory, exist_ok=True)

print("\n✅ تم إنشاء بنية المجلدات!")

🔍 التحقق من المسارات...
   صور التدريب: ✅ /kaggle/input/competitions/dal-shemagh-detection-challenge/images/train
   Labels التدريب: ✅ /kaggle/input/competitions/dal-shemagh-detection-challenge/labels/train
   صور الاختبار: ✅ /kaggle/input/competitions/dal-shemagh-detection-challenge/images/test

✅ تم إنشاء بنية المجلدات!


In [4]:
# فحص البيانات
train_images = glob.glob(f"{TRAIN_IMAGES_SOURCE}/*.jpg") + glob.glob(f"{TRAIN_IMAGES_SOURCE}/*.png")
train_labels = glob.glob(f"{TRAIN_LABELS_SOURCE}/*.txt")
test_images = glob.glob(f"{TEST_IMAGES_SOURCE}/*.jpg") + glob.glob(f"{TEST_IMAGES_SOURCE}/*.png")

print(f"📈 إحصائيات البيانات:")
print(f"   📸 صور التدريب: {len(train_images)}")
print(f"   📝 ملفات Labels: {len(train_labels)}")
print(f"   🧪 صور الاختبار: {len(test_images)}")

# تحليل الـ labels
class_counts = {0: 0, 1: 0}
empty_labels = 0

for label_file in train_labels:
    with open(label_file, 'r') as f:
        lines = f.readlines()
        if len(lines) == 0:
            empty_labels += 1
        else:
            for line in lines:
                if line.strip():
                    class_id = int(line.split()[0])
                    class_counts[class_id] += 1

print(f"\n📊 توزيع الـ Classes:")
print(f"   👤 Head (Class 0): {class_counts[0]}")
print(f"   🧣 Shemagh (Class 1): {class_counts[1]}")
print(f"   📭 صور فارغة: {empty_labels}")

📈 إحصائيات البيانات:
   📸 صور التدريب: 651
   📝 ملفات Labels: 651
   🧪 صور الاختبار: 842

📊 توزيع الـ Classes:
   👤 Head (Class 0): 446
   🧣 Shemagh (Class 1): 165
   📭 صور فارغة: 203


In [5]:
def split_dataset(images_source, labels_source, train_ratio=0.8):
    image_files = glob.glob(f"{images_source}/*.jpg") + glob.glob(f"{images_source}/*.png")
    
    np.random.seed(42)
    np.random.shuffle(image_files)
    
    split_idx = int(len(image_files) * train_ratio)
    train_images = image_files[:split_idx]
    val_images = image_files[split_idx:]
    
    print(f"📚 صور التدريب: {len(train_images)}")
    print(f"📖 صور التحقق: {len(val_images)}")
    
    # نسخ ملفات التدريب
    for img_path in tqdm(train_images, desc="نسخ ملفات التدريب"):
        filename = os.path.basename(img_path)
        label_name = os.path.splitext(filename)[0] + '.txt'
        
        shutil.copy(img_path, f"{TRAIN_IMAGES_DIR}/{filename}")
        
        label_path = f"{labels_source}/{label_name}"
        if os.path.exists(label_path):
            shutil.copy(label_path, f"{TRAIN_LABELS_DIR}/{label_name}")
        else:
            open(f"{TRAIN_LABELS_DIR}/{label_name}", 'w').close()
    
    # نسخ ملفات التحقق
    for img_path in tqdm(val_images, desc="نسخ ملفات التحقق"):
        filename = os.path.basename(img_path)
        label_name = os.path.splitext(filename)[0] + '.txt'
        
        shutil.copy(img_path, f"{VAL_IMAGES_DIR}/{filename}")
        
        label_path = f"{labels_source}/{label_name}"
        if os.path.exists(label_path):
            shutil.copy(label_path, f"{VAL_LABELS_DIR}/{label_name}")
        else:
            open(f"{VAL_LABELS_DIR}/{label_name}", 'w').close()
    
    print("✅ تم تقسيم البيانات بنجاح!")

# تنفيذ التقسيم
split_dataset(TRAIN_IMAGES_SOURCE, TRAIN_LABELS_SOURCE)

📚 صور التدريب: 520
📖 صور التحقق: 131


نسخ ملفات التحقق: 100%|██████████| 131/131 [00:00<00:00, 132.97it/s]

✅ تم تقسيم البيانات بنجاح!


ل كل شي صحيح هنا ؟ و ليس سويت كذا مع انه عندنا بيانات اختبار , فالبيانات مقسمه من نفس المسابقه لتدريب و اختبار , ف ليش رجعت قسمت بيانات التدريب لنوعين و هل هذا صح ؟      -سؤال اسله ال اذكاء عن الكود الي فوقي مباشرة- وبس 

In [6]:
data_yaml = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 2,
    'names': ['head', 'shemagh']
}

yaml_path = f"{DATASET_DIR}/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✅ تم إنشاء ملف data.yaml")

✅ تم إنشاء ملف data.yaml


In [7]:
# تحميل النموذج
model = YOLO('yolov8n.pt')

# التدريب
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name='shemagh_model',
    patience=20,
    save=True,
    device=0,
    workers=2,
    project=f'{WORKING_DIR}/runs/detect',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    seed=42,
    cos_lr=True,
    amp=True
)

print("✅ انتهى التدريب!")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=shemagh_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20,

In [8]:
def calculate_iou(box1, box2):
    """حساب IoU بين bounding boxes"""
    box1_x1 = box1[0] - box1[2] / 2
    box1_y1 = box1[1] - box1[3] / 2
    box1_x2 = box1[0] + box1[2] / 2
    box1_y2 = box1[1] + box1[3] / 2
    
    box2_x1 = box2[0] - box2[2] / 2
    box2_y1 = box2[1] - box2[3] / 2
    box2_x2 = box2[0] + box2[2] / 2
    box2_y2 = box2[1] + box2[3] / 2
    
    inter_x1 = max(box1_x1, box2_x1)
    inter_y1 = max(box1_y1, box2_y1)
    inter_x2 = min(box1_x2, box2_x2)
    inter_y2 = min(box1_y2, box2_y2)
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    
    union_area = box1_area + box2_area - inter_area
    
    iou = inter_area / union_area if union_area > 0 else 0
    return iou

def determine_right_place(predictions, iou_threshold=0.3):
    """تحديد هل الشماغ في المكان الصحيح"""
    heads = []
    shemaghs = []
    
    for pred in predictions:
        class_id = int(pred[0])
        bbox = pred[2:6]
        
        if class_id == 0:
            heads.append(bbox)
        elif class_id == 1:
            shemaghs.append(bbox)
    
    if len(heads) == 0 or len(shemaghs) == 0:
        return 0
    
    max_iou = 0
    for head in heads:
        for shemagh in shemaghs:
            iou = calculate_iou(head, shemagh)
            max_iou = max(max_iou, iou)
    
    return 1 if max_iou >= iou_threshold else 0

print("✅ تم تعريف الدوال بنجاح!")

✅ تم تعريف الدوال بنجاح!


In [9]:
!pip install ultralytics -q
import os
import glob
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

def calculate_iou(box1, box2):
    box1_x1 = box1[0] - box1[2] / 2
    box1_y1 = box1[1] - box1[3] / 2
    box1_x2 = box1[0] + box1[2] / 2
    box1_y2 = box1[1] + box1[3] / 2
    
    box2_x1 = box2[0] - box2[2] / 2
    box2_y1 = box2[1] - box2[3] / 2
    box2_x2 = box2[0] + box2[2] / 2
    box2_y2 = box2[1] + box2[3] / 2
    
    inter_x1 = max(box1_x1, box2_x1)
    inter_y1 = max(box1_y1, box2_y1)
    inter_x2 = min(box1_x2, box2_x2)
    inter_y2 = min(box1_y2, box2_y2)
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0

def determine_right_place(predictions, iou_threshold=0.3):
    heads = []
    shemaghs = []
    
    for pred in predictions:
        class_id = int(pred[0])
        bbox = pred[2:6]
        if class_id == 0:
            heads.append(bbox)
        elif class_id == 1:
            shemaghs.append(bbox)
    
    if len(heads) == 0 or len(shemaghs) == 0:
        return 0
    
    max_iou = 0
    for head in heads:
        for shemagh in shemaghs:
            iou = calculate_iou(head, shemagh)
            max_iou = max(max_iou, iou)
    
    return 1 if max_iou >= iou_threshold else 0

# ===== المسارات مباشرة =====
MODEL_PATH = "/kaggle/working/runs/detect/shemagh_model/weights/best.pt"
TEST_DIR = "/kaggle/input/competitions/dal-shemagh-detection-challenge/images/test"
OUTPUT_CSV = "/kaggle/working/submission.csv"

print("🔮 بدء التنبؤ على صور الاختبار...")

model = YOLO(MODEL_PATH)

test_images = sorted(
    glob.glob(f"{TEST_DIR}/*.jpg") + 
    glob.glob(f"{TEST_DIR}/*.png")
)

print(f"📸 عدد صور الاختبار: {len(test_images)}")

results_list = []

for img_path in tqdm(test_images, desc="معالجة الصور"):
    filename = os.path.basename(img_path)
    
    results = model.predict(img_path, conf=0.25, iou=0.45, verbose=False)
    
    predictions = []
    prediction_string_parts = []
    
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        
        for i in range(len(boxes)):
            class_id = int(boxes.cls[i].item())
            confidence = float(boxes.conf[i].item())
            x_center = float(boxes.xywhn[i][0].item())
            y_center = float(boxes.xywhn[i][1].item())
            width = float(boxes.xywhn[i][2].item())
            height = float(boxes.xywhn[i][3].item())
            
            predictions.append([class_id, confidence, x_center, y_center, width, height])
            pred_str = f"{class_id} {confidence:.4f} {x_center:.4f} {y_center:.4f} {width:.4f} {height:.4f}"
            prediction_string_parts.append(pred_str)
    
    right_place = determine_right_place(predictions) if predictions else 0
    prediction_string = " ".join(prediction_string_parts) if prediction_string_parts else "-"
    
    results_list.append({
        'filename': filename,
        'right_place': right_place,
        'prediction_string': prediction_string
    })

df = pd.DataFrame(results_list)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ تم حفظ submission.csv")
print(f"📊 عدد الصور: {len(df)}")
print(f"✔️  الشماغ صح: {df['right_place'].sum()}")
print(f"❌ الشماغ غلط: {len(df) - df['right_place'].sum()}")
print(f"\n📋 عينة:")
print(df.head(5))

🔮 بدء التنبؤ على صور الاختبار...
📸 عدد صور الاختبار: 842


معالجة الصور: 100%|██████████| 842/842 [00:15<00:00, 53.02it/s]


✅ تم حفظ submission.csv
📊 عدد الصور: 842
✔️  الشماغ صح: 31
❌ الشماغ غلط: 811

📋 عينة:
  filename  right_place                                  prediction_string
0    0.jpg            0               0 0.8773 0.4477 0.1083 0.2434 0.2107
1    1.jpg            0               1 0.8119 0.3555 0.4875 0.6966 0.9584
2   10.jpg            0                                                  -
3  100.jpg            0                                                  -
4  101.jpg            0  1 0.7715 0.2732 0.6250 0.5464 0.6615 0 0.2746 ...


تحسين النموذج 1

In [10]:
# تثبيت المكتبات
!pip install -q ultralytics albumentations

print("✅ تم تثبيت المكتبات!")

✅ تم تثبيت المكتبات!


In [11]:
import os
import yaml
from ultralytics import YOLO
import albumentations as A

# ===== Data Augmentation =====
my_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    A.GaussianBlur(p=0.1),
    A.Rotate(limit=15),
])

# ===== إعداد data.yaml =====
BASE_INPUT = "/kaggle/input/competitions/dal-shemagh-detection-challenge"

data_yaml = {
    'path': BASE_INPUT,
    'train': 'images/train',
    'val': 'images/train',
    'nc': 2,
    'names': {0: 'head', 1: 'shemagh'}
}

yaml_path = '/kaggle/working/data_v2.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ تم إنشاء data.yaml!")

# ===== تحميل النموذج الجديد =====
print("🚀 بدء التدريب بالنموذج المحسّن...")
print("   النموذج: yolo11n.pt (أحدث وأفضل!)")
print("   Epochs: 150")
print("   IoU threshold: 0.1")

model_v2 = YOLO('yolo11n.pt')

results_v2 = model_v2.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,
    name='shemagh_model_v2',
    patience=30,
    save=True,
    device=0,
    workers=2,
    project='/kaggle/working/runs/detect',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    seed=42,
    cos_lr=True,
    amp=True
)

print("✅ انتهى التدريب!")

✅ تم إنشاء data.yaml!
🚀 بدء التدريب بالنموذج المحسّن...
   النموذج: yolo11n.pt (أحدث وأفضل!)
   Epochs: 150
   IoU threshold: 0.1
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=

In [13]:
import glob
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# ===== دوال IoU =====
def calculate_iou(box1, box2):
    b1_x1, b1_x2 = box1[0] - box1[2]/2, box1[0] + box1[2]/2
    b1_y1, b1_y2 = box1[1] - box1[3]/2, box1[1] + box1[3]/2
    
    b2_x1, b2_x2 = box2[0] - box2[2]/2, box2[0] + box2[2]/2
    b2_y1, b2_y2 = box2[1] - box2[3]/2, box2[1] + box2[3]/2
    
    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0

# ===== المسارات =====
MODEL_PATH_V2 = "/kaggle/working/runs/detect/shemagh_model_v2/weights/best.pt"
TEST_DIR = "/kaggle/input/competitions/dal-shemagh-detection-challenge/images/test"
OUTPUT_CSV = "/kaggle/working/submission_v2.csv"

print("🔮 بدء التنبؤ...")

model_v2 = YOLO(MODEL_PATH_V2)

test_images = sorted(
    glob.glob(f"{TEST_DIR}/*.jpg") +
    glob.glob(f"{TEST_DIR}/*.png")
)

print(f"📸 عدد صور الاختبار: {len(test_images)}")

results_list = []

for img_path in tqdm(test_images, desc="معالجة الصور"):
    filename = os.path.basename(img_path)
    results = model_v2.predict(img_path, conf=0.25, iou=0.45, verbose=False)
    
    prediction_strings = []
    person_boxes = []
    shemagh_boxes = []
    
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        
        if len(boxes) == 0:
            prediction_str = "-"
            right_place = 0
        else:
            for box in boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                x, y, w, h = box.xywhn[0].tolist()
                
                prediction_strings.append(
                    f"{cls} {conf:.4f} {x:.4f} {y:.4f} {w:.4f} {h:.4f}"
                )
                
                if cls == 0:
                    person_boxes.append([x, y, w, h])
                elif cls == 1:
                    shemagh_boxes.append([x, y, w, h])
            
            prediction_str = " ".join(prediction_strings)
            
            # IoU threshold = 0.1 (محسّن!)
            right_place = 0
            if person_boxes and shemagh_boxes:
                for s_box in shemagh_boxes:
                    for p_box in person_boxes:
                        iou = calculate_iou(s_box, p_box)
                        if iou > 0.1:
                            right_place = 1
                            break
                    if right_place == 1:
                        break
    else:
        prediction_str = "-"
        right_place = 0
    
    results_list.append({
        'filename': filename,
        'right_place': right_place,
        'prediction_string': prediction_str
    })

df = pd.DataFrame(results_list)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ تم حفظ submission_v2.csv")
print(f"📊 عدد الصور: {len(df)}")
print(f"✔️  الشماغ صح: {df['right_place'].sum()}")
print(f"❌ الشماغ غلط: {len(df) - df['right_place'].sum()}")
print(df.head(5))

🔮 بدء التنبؤ...
📸 عدد صور الاختبار: 842


معالجة الصور: 100%|██████████| 842/842 [00:12<00:00, 68.49it/s]


✅ تم حفظ submission_v2.csv
📊 عدد الصور: 842
✔️  الشماغ صح: 83
❌ الشماغ غلط: 759
  filename  right_place                     prediction_string
0    0.jpg            0  0 0.9079 0.4419 0.1096 0.2341 0.2039
1    1.jpg            0  1 0.7046 0.3109 0.4646 0.6114 0.9287
2   10.jpg            0                                     -
3  100.jpg            0                                     -
4  101.jpg            0  1 0.8565 0.2651 0.6463 0.5272 0.7073


التحسين 2 

In [2]:
!pip install -q ultralytics albumentations

import os
import yaml
import glob
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# ===== Data Augmentation =====
import albumentations as A
my_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    A.GaussianBlur(p=0.2),
    A.Rotate(limit=20),
    A.RandomRain(p=0.1),
])

# ===== data.yaml =====
BASE_INPUT = "/kaggle/input/competitions/dal-shemagh-detection-challenge"

data_yaml = {
    'path': BASE_INPUT,
    'train': 'images/train',
    'val': 'images/train',  # كل البيانات للتدريب!
    'nc': 2,
    'names': {0: 'head', 1: 'shemagh'}
}

yaml_path = '/kaggle/working/data_v3.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ تم إنشاء data.yaml!")

# ===== تدريب النموذج المحسّن =====
print("🚀 بدء التدريب...")
model_v3 = YOLO('yolo11s.pt') 
results_v3 = model_v3.train(
    data=yaml_path,
    epochs=200,
    imgsz=640,
    batch=16,
    name='shemagh_model_v3',
    patience=40,
    save=True,
    device=0,
    workers=2,
    project='/kaggle/working/runs/detect',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,           # ← جديد
    lrf=0.0001,          # ← جديد
    weight_decay=0.0005, # ← جديد
    seed=42,
    cos_lr=True,
    amp=True
)


print("✅ انتهى التدريب!")

✅ تم إنشاء data.yaml!
🚀 بدء التدريب...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data_v3.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.0001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=shemagh_model_v3, nbs=64, nms=False, opset=None, optimize=False, optimizer=Ada

In [3]:
import glob
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
import os

def calculate_iou(box1, box2):
    b1_x1, b1_x2 = box1[0] - box1[2]/2, box1[0] + box1[2]/2
    b1_y1, b1_y2 = box1[1] - box1[3]/2, box1[1] + box1[3]/2
    b2_x1, b2_x2 = box2[0] - box2[2]/2, box2[0] + box2[2]/2
    b2_y1, b2_y2 = box2[1] - box2[3]/2, box2[1] + box2[3]/2
    
    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0

MODEL_PATH_V3 = "/kaggle/working/runs/detect/shemagh_model_v3/weights/best.pt"
TEST_DIR = "/kaggle/input/competitions/dal-shemagh-detection-challenge/images/test"
OUTPUT_CSV = "/kaggle/working/submission_v3.csv"

print("🔮 بدء التنبؤ...")

model_v3 = YOLO(MODEL_PATH_V3)

test_images = sorted(
    glob.glob(f"{TEST_DIR}/*.jpg") +
    glob.glob(f"{TEST_DIR}/*.png")
)

print(f"📸 عدد صور الاختبار: {len(test_images)}")

results_list = []

for img_path in tqdm(test_images, desc="معالجة الصور"):
    filename = os.path.basename(img_path)
    results = model_v3.predict(img_path, conf=0.15, iou=0.45, verbose=False)
    
    prediction_strings = []
    person_boxes = []
    shemagh_boxes = []
    
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        
        if len(boxes) == 0:
            prediction_str = "-"
            right_place = 0
        else:
            for box in boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                x, y, w, h = box.xywhn[0].tolist()
                
                prediction_strings.append(
                    f"{cls} {conf:.4f} {x:.4f} {y:.4f} {w:.4f} {h:.4f}"
                )
                
                if cls == 0:
                    person_boxes.append([x, y, w, h])
                elif cls == 1:
                    shemagh_boxes.append([x, y, w, h])
            
            prediction_str = " ".join(prediction_strings)
            
            right_place = 0
            if person_boxes and shemagh_boxes:
                for s_box in shemagh_boxes:
                    for p_box in person_boxes:
                        iou = calculate_iou(s_box, p_box)
                        if iou > 0.1:
                            right_place = 1
                            break
                    if right_place == 1:
                        break
    else:
        prediction_str = "-"
        right_place = 0
    
    results_list.append({
        'filename': filename,
        'right_place': right_place,
        'prediction_string': prediction_str
    })

df = pd.DataFrame(results_list)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ تم حفظ submission_v3.csv")
print(f"📊 عدد الصور: {len(df)}")
print(f"✔️  الشماغ صح: {df['right_place'].sum()}")
print(f"❌ الشماغ غلط: {len(df) - df['right_place'].sum()}")
print(df.head(5))

🔮 بدء التنبؤ...
📸 عدد صور الاختبار: 842


معالجة الصور: 100%|██████████| 842/842 [00:19<00:00, 43.25it/s]



✅ تم حفظ submission_v3.csv
📊 عدد الصور: 842
✔️  الشماغ صح: 122
❌ الشماغ غلط: 720
  filename  right_place                     prediction_string
0    0.jpg            0  0 0.9408 0.4354 0.1117 0.2244 0.2041
1    1.jpg            0  1 0.9013 0.3111 0.5006 0.6213 0.9988
2   10.jpg            0                                     -
3  100.jpg            0  1 0.8075 0.5038 0.5780 0.9924 0.8439
4  101.jpg            0  1 0.9179 0.2657 0.6432 0.5313 0.7132
